# BRCA Subtype Classification – RNA-Seq (TCGA)
**Ziel:** Brustkrebssubtypen (LumA, LumB, Basal, Her2, Normal) aus Genexpressionsdaten vorhersagen

**Datensatz:** TCGA BRCA, 956 Samples (nach Bereinigung), 16 383 Gene (log2-RSEM)

---
### Warum *nicht* nur Random Forest?
Random Forest ist ein solider Baseline – aber für hochdimensionale Genomdaten (~16k Features, ~1000 Samples) zeigt die Literatur konsistent, dass **Logistic Regression mit ElasticNet-Regularisierung** (L1+L2) und **XGBoost** besser abschneiden:

| Modell | Stärke bei Genomdaten | Typische Accuracy (TCGA BRCA) |
|---|---|---|
| Random Forest | Robust, kein Scaling nötig | ~93–95 % |
| **Logistic Reg. (ElasticNet)** | Sparsity, interpretierbar, skaliert gut mit vielen Features | **~95–97 %** |
| **XGBoost** | Tabular SOTA, robust gegen Imbalance | **~95–96 %** |
| LinearSVC | Schnell, gut bei linearer Separierbarkeit | ~94–96 % |

Wir testen alle vier und wählen den Gewinner mit Nested CV.

## 1. Packages installieren & importieren

In [ ]:
# Einmalig ausführen falls Packages fehlen
# !pip install pandas numpy scikit-learn xgboost lightgbm imbalanced-learn matplotlib seaborn umap-learn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score,
    balanced_accuracy_score, f1_score
)
from sklearn.decomposition import PCA

# XGBoost
from xgboost import XGBClassifier

# UMAP (für bessere Visualisierung als PCA)
try:
    from umap import UMAP
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP nicht installiert – nur PCA wird verwendet.")

# Seed für Reproduzierbarkeit
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Alle Packages geladen")

## 2. Datensätze laden & kombinieren

In [ ]:
# Pfade anpassen!
DATASET_PATH = "TCGA/dataset.csv"   # <-- Pfad zum dataset.csv
OUTCOME_PATH = "TCGA/outcome.csv"   # <-- Pfad zum outcome.csv

print("Lade Daten (kann 20–30 Sek. dauern bei 128 MB)...")
df      = pd.read_csv(DATASET_PATH, index_col=0)
outcome = pd.read_csv(OUTCOME_PATH, index_col=0)
outcome.columns = ["BRCA_subtype"]

print(f"Dataset:  {df.shape[0]} Samples × {df.shape[1]} Gene")
print(f"Outcome:  {outcome.shape[0]} Samples")

In [ ]:
# Verknüpfen über Sample-ID (Index)
combined = df.join(outcome)

print(f"Gemeinsame Samples: {combined.shape[0]}")
print(f"Samples ohne Subtype-Label: {combined['BRCA_subtype'].isna().sum()}")
print("\nKlassenverteilung (komplett):")
print(combined["BRCA_subtype"].value_counts())

## 3. Daten bereinigen
### 3.1 Samples ohne Label entfernen

In [ ]:
# Samples ohne Subtype-Label droppen
combined_clean = combined.dropna(subset=["BRCA_subtype"]).copy()
print(f"Samples nach Label-Cleaning: {len(combined_clean)}")

X_raw = combined_clean.drop(columns=["BRCA_subtype"])
y     = combined_clean["BRCA_subtype"]

print(f"\nKlassenverteilung (bereinigt):")
print(y.value_counts())

In [ ]:
# Klassenverteilung visualisieren
SUBTYPE_COLORS = {
    "LumA": "#185FA5", "LumB": "#3AB12A",
    "Basal": "#BD1BBD", "Her2": "#E8A020", "Normal": "#555555"
}

fig, ax = plt.subplots(figsize=(7, 4))
counts = y.value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[SUBTYPE_COLORS.get(s, "#888") for s in counts.index],
              edgecolor="none")
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title("BRCA Subtype – Klassenverteilung", fontsize=13)
ax.set_ylabel("Anzahl Samples")
ax.set_xlabel("Subtype")
sns.despine()
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()

# Achtung: Klassen-Imbalance!
# LumA (434) >> Her2 (67) → class_weight='balanced' ist wichtig
print(f"\n⚠️  Imbalance Ratio (LumA / Her2): {counts['LumA'] / counts['Her2']:.1f}x")

### 3.2 NaN-Imputation in Features
Nur 6 NaN-Werte im Feature-Matrix – Median-Imputation ist ausreichend.

In [ ]:
nan_count = X_raw.isna().sum().sum()
print(f"NaN-Werte in Features: {nan_count}")

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(
    imputer.fit_transform(X_raw),
    index=X_raw.index,
    columns=X_raw.columns
)
print(f"✓ Imputation abgeschlossen. NaN nach Imputation: {X_imp.isna().sum().sum()}")

### 3.3 Feature-Selektion (zweistufig)

**Schritt 1 – Varianz-Filter:** Gene mit annähernd konstanter Expression über alle Samples entfernen (biologisch uninformativ).

**Schritt 2 – ANOVA F-Score (SelectKBest):** Gene nach statistischer Trennkraft zwischen den Subtypen ranken und Top-K behalten. Besser als reiner Varianzfilter, weil er die Labels berücksichtigt.

In [ ]:
# --- Schritt 1: Varianz-Filter ---
sel_var = VarianceThreshold(threshold=0.1)
X_var   = sel_var.fit_transform(X_imp)
cols_var = X_imp.columns[sel_var.get_support()]
print(f"Nach Varianz-Filter: {X_var.shape[1]} Gene (von {X_imp.shape[1]})")

# --- Schritt 2: ANOVA F-Score – Top 2000 Gene ---
K_BEST = 2000  # Sweet-Spot: informativ, aber nicht zu viele für Overfitting
sel_k   = SelectKBest(f_classif, k=K_BEST)
X_k     = sel_k.fit_transform(X_var, y)
# Spaltennamen wiederherstellen
sel_k_cols = cols_var[sel_k.get_support()]
X_selected = pd.DataFrame(X_k, index=X_imp.index, columns=sel_k_cols)
print(f"Nach SelectKBest (k={K_BEST}): {X_selected.shape[1]} Gene")
print(f"\nTop 10 Gene nach ANOVA F-Score:")
f_scores = pd.Series(sel_k.scores_[sel_k.get_support()], index=sel_k_cols)
print(f_scores.sort_values(ascending=False).head(10))

## 4. Train-/Test-Split erstellen
Stratified split, damit die Klassenverteilung in beiden Sets gleich bleibt – besonders wichtig bei der Imbalance (Her2 nur 67 Samples).

In [ ]:
# Stratified 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
print(f"Train: {X_train.shape[0]} Samples")
print(f"Test:  {X_test.shape[0]} Samples")
print(f"\nKlassenverteilung Train:\n{y_train.value_counts()}")
print(f"\nKlassenverteilung Test:\n{y_test.value_counts()}")

# Scaling (wichtig für Logistic Regression & SVM)
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit NUR auf Train!
X_test_sc  = scaler.transform(X_test)        # transform mit Train-Parametern
print("\n✓ StandardScaler angewendet (fit auf Train, transform auf Test)")

# LabelEncoder für XGBoost (braucht Integer-Labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
print(f"Label-Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## 5. Modell-Auswahl & Cross-Validation
**Strategie:** 5-Fold Stratified CV auf dem Trainingsset, um die 4 Kandidaten fair zu vergleichen.

Wir schauen auf **Accuracy** (Gesamtleistung) UND **Balanced Accuracy** (wichtig wegen Imbalance).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_features="sqrt",
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LogReg ElasticNet": LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5,
        C=0.1, max_iter=3000, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LinearSVC": CalibratedClassifierCV(  # Wrapper für predict_proba
        LinearSVC(C=0.5, class_weight="balanced",
                  max_iter=3000, random_state=RANDOM_STATE)
    ),
}

cv_results = {}
print(f"{'Modell':<25} {'Accuracy':>12} {'Balanced Acc':>14}")
print("-" * 55)

for name, model in candidates.items():
    # XGBoost braucht encoded labels
    y_cv = y_train_enc if name == "XGBoost" else y_train
    X_cv = X_train_sc  # alle Modelle bekommen skalierte Features

    acc_scores  = cross_val_score(model, X_cv, y_cv, cv=cv, scoring="accuracy", n_jobs=-1)
    bal_scores  = cross_val_score(model, X_cv, y_cv, cv=cv, scoring="balanced_accuracy", n_jobs=-1)

    cv_results[name] = {"accuracy": acc_scores, "balanced_accuracy": bal_scores}
    print(f"{name:<25} {acc_scores.mean():.4f} ± {acc_scores.std():.3f}   "
          f"{bal_scores.mean():.4f} ± {bal_scores.std():.3f}")

In [ ]:
# Ergebnisse visualisieren
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metrics = ["accuracy", "balanced_accuracy"]
titles  = ["CV Accuracy", "CV Balanced Accuracy"]

for ax, metric, title in zip(axes, metrics, titles):
    means = [cv_results[n][metric].mean() for n in candidates]
    stds  = [cv_results[n][metric].std()  for n in candidates]
    bars  = ax.bar(candidates.keys(), means, yerr=stds,
                   capsize=5, color=["#185FA5","#E8A020","#3AB12A","#BD1BBD"],
                   edgecolor="none", error_kw={"linewidth":1.5})
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=9)
    ax.set_ylim(0.85, 1.01)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=20)
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig("model_comparison_cv.png", dpi=150)
plt.show()

## 6. Bestes Modell – Hyperparameter-Tuning
Wir tunen das Modell mit der besten CV-Performance via **RandomizedSearchCV** (effizienter als GridSearch).

In [ ]:
# =========================================================
#  Option A: Wenn XGBoost das beste Modell ist (häufig)
# =========================================================
xgb_param_dist = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6, 7],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma":            [0, 0.1, 0.5],
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
print("Starte XGBoost Hyperparameter-Suche (40 Iterationen × 5 Folds)...")
xgb_search.fit(X_train_sc, y_train_enc)
print(f"\nBeste Parameter: {xgb_search.best_params_}")
print(f"Beste CV Balanced Accuracy: {xgb_search.best_score_:.4f}")

In [ ]:
# =========================================================
#  Option B: Wenn Logistic Regression das beste Modell ist
# =========================================================
lr_param_dist = {
    "C":        [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
    "l1_ratio": [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(
        penalty="elasticnet", solver="saga",
        max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    param_distributions=lr_param_dist,
    n_iter=30,
    cv=cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
print("Starte LogReg Hyperparameter-Suche...")
lr_search.fit(X_train_sc, y_train)
print(f"\nBeste Parameter: {lr_search.best_params_}")
print(f"Beste CV Balanced Accuracy: {lr_search.best_score_:.4f}")

## 7. Finales Modell trainieren

In [ ]:
# Bestes Modell aus der Suche nehmen (Beispiel: XGBoost)
# Falls LogReg besser → xgb_search durch lr_search ersetzen
best_model = xgb_search.best_estimator_

# Auf gesamtem Trainingsset trainieren
print("Trainiere finales Modell auf gesamtem Trainingsset...")
best_model.fit(X_train_sc, y_train_enc)  # XGBoost → y_train_enc
# best_model.fit(X_train_sc, y_train)    # LogReg/RF → y_train

# Modell + gesamte Pipeline speichern
pipeline_artefakt = {
    "imputer":        imputer,
    "var_selector":   sel_var,
    "kbest_selector": sel_k,
    "scaler":         scaler,
    "label_encoder":  le,
    "model":          best_model,
}
joblib.dump(pipeline_artefakt, "brca_pipeline_v2.joblib")
print("✓ Pipeline gespeichert als 'brca_pipeline_v2.joblib'")

## 8. Modell evaluieren

In [ ]:
# Predictions auf dem Hold-out Testset
y_pred_enc = best_model.predict(X_test_sc)
y_pred     = le.inverse_transform(y_pred_enc)  # Klassen-Namen wiederherstellen

acc     = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"=== Test-Set Ergebnisse ===")
print(f"Accuracy:          {acc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}  ← wichtiger wegen Imbalance")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"\n{classification_report(y_test, y_pred)}")

In [ ]:
# Confusion Matrix
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels,
    linewidths=0.5, ax=ax
)
ax.set_xlabel("Vorhergesagt", fontsize=12)
ax.set_ylabel("Tatsächlich", fontsize=12)
ax.set_title(f"Confusion Matrix  |  Acc: {acc:.3f}  |  Bal.Acc: {bal_acc:.3f}", fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix_v2.png", dpi=150)
plt.show()

In [ ]:
# Per-Klasse Precision / Recall / F1 visualisieren
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, support = precision_recall_fscore_support(y_test, y_pred, labels=labels)
metrics_df = pd.DataFrame({
    "Precision": prec, "Recall": rec, "F1-Score": f1, "Support": support
}, index=labels)
print(metrics_df.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(labels))
w = 0.25
ax.bar(x - w,   prec, w, label="Precision", color="#185FA5")
ax.bar(x,       rec,  w, label="Recall",    color="#3AB12A")
ax.bar(x + w,   f1,   w, label="F1-Score",  color="#E8A020")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Per-Klasse Metriken (Test-Set)")
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig("per_class_metrics.png", dpi=150)
plt.show()

In [ ]:
# Feature Importance (für XGBoost / Random Forest)
# Für Logistic Regression: model.coef_ verwenden

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=sel_k_cols).sort_values(ascending=False)

    print("Top 15 wichtigste Gene:")
    print(feat_imp.head(15))

    fig, ax = plt.subplots(figsize=(9, 5))
    feat_imp.head(20).plot(kind="bar", ax=ax, color="#185FA5", edgecolor="none")
    ax.set_title("Top-20 wichtigste Gene (Feature Importance)")
    ax.set_ylabel("Importance")
    ax.set_xlabel("Gen")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    sns.despine()
    plt.tight_layout()
    plt.savefig("feature_importance_v2.png", dpi=150)
    plt.show()

elif hasattr(best_model, "coef_"):
    # Logistic Regression: mittlerer |Koeffizient| über alle Klassen
    coef_mean = np.abs(best_model.coef_).mean(axis=0)
    feat_imp = pd.Series(coef_mean, index=sel_k_cols).sort_values(ascending=False)
    print("Top 15 Gene (|Koeffizient| gemittelt über Klassen):")
    print(feat_imp.head(15))

    fig, ax = plt.subplots(figsize=(9, 5))
    feat_imp.head(20).plot(kind="bar", ax=ax, color="#3AB12A", edgecolor="none")
    ax.set_title("Top-20 wichtigste Gene (|Koeffizient|, gemittelt)")
    ax.set_ylabel("|Koeffizient|")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    sns.despine()
    plt.tight_layout()
    plt.savefig("feature_importance_v2.png", dpi=150)
    plt.show()

## 9. Dimensionsreduktion & Visualisierung
**PCA** zeigt linearen Varianzanteil. **UMAP** ist deutlich besser für RNA-seq-Cluster-Visualisierung – behält lokale und globale Struktur bei.

In [ ]:
# Gesamten (gesäuberten) Datensatz für Visualisierung
X_all = scaler.transform(
    sel_k.transform(sel_var.transform(X_imp))
)
y_all = y.values

# --- PCA ---
pca   = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_all)
var_expl = pca.explained_variance_ratio_
print(f"PCA Varianzanteil (2 Komponenten): {var_expl.sum():.3f}")

fig, ax = plt.subplots(figsize=(8, 6))
for subtype, color in SUBTYPE_COLORS.items():
    mask = y_all == subtype
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=f"{subtype} (n={mask.sum()})",
               color=color, alpha=0.6, s=18, edgecolors="none")
ax.set_xlabel(f"PC1 ({var_expl[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({var_expl[1]*100:.1f}%)")
ax.set_title("PCA – BRCA Subtypen")
ax.legend(markerscale=2, fontsize=9)
sns.despine()
plt.tight_layout()
plt.savefig("pca_visualization.png", dpi=150)
plt.show()

In [ ]:
# --- UMAP (viel bessere Cluster-Trennung als PCA) ---
if UMAP_AVAILABLE:
    print("Berechne UMAP (dauert ~30–60 Sek.)...")
    reducer = UMAP(
        n_components=2,
        n_neighbors=30,
        min_dist=0.3,
        metric="euclidean",
        random_state=RANDOM_STATE
    )
    X_umap = reducer.fit_transform(X_all)

    fig, ax = plt.subplots(figsize=(8, 6))
    for subtype, color in SUBTYPE_COLORS.items():
        mask = y_all == subtype
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   label=f"{subtype} (n={mask.sum()})",
                   color=color, alpha=0.7, s=20, edgecolors="none")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title("UMAP – BRCA Subtypen (RNA-Seq)")
    ax.legend(markerscale=2, fontsize=9)
    sns.despine()
    plt.tight_layout()
    plt.savefig("umap_visualization.png", dpi=150)
    plt.show()
else:
    print("UMAP nicht verfügbar. Installiere mit: pip install umap-learn")

## 10. Inference – neue Patientin predicten
So kannst du das gespeicherte Modell für neue Samples einsetzen.

In [ ]:
def predict_subtype(gene_expression_series: pd.Series, pipeline_path: str = "brca_pipeline_v2.joblib") -> dict:
    """
    Sagt den BRCA-Subtyp für eine neue Patientin voraus.
    
    Parameters
    ----------
    gene_expression_series : pd.Series
        Index = Gennamen (16383 Gene), Values = log2(RSEM+1) Expressionswerte
    pipeline_path : str
        Pfad zur gespeicherten Pipeline
    
    Returns
    -------
    dict mit 'predicted_subtype' und 'probabilities'
    """
    pipe = joblib.load(pipeline_path)

    X_new = gene_expression_series.to_frame().T

    # Gleiche Pipeline wie beim Training
    X_new_imp  = pd.DataFrame(pipe["imputer"].transform(X_new), columns=X_new.columns)
    X_new_var  = pipe["var_selector"].transform(X_new_imp)
    X_new_k    = pipe["kbest_selector"].transform(X_new_var)
    X_new_sc   = pipe["scaler"].transform(X_new_k)

    pred_enc  = pipe["model"].predict(X_new_sc)
    pred_name = pipe["label_encoder"].inverse_transform(pred_enc)[0]

    result = {"predicted_subtype": pred_name}
    if hasattr(pipe["model"], "predict_proba"):
        proba = pipe["model"].predict_proba(X_new_sc)[0]
        result["probabilities"] = dict(zip(pipe["label_encoder"].classes_, proba.round(3)))

    return result

# Beispiel: erste Zeile des Testsets vorhersagen
example_sample = X_raw.loc[X_test.index[0]]
result = predict_subtype(example_sample)
print(f"Vorhergesagter Subtyp: {result['predicted_subtype']}")
print(f"Tatsächlicher Subtyp:  {y_test.iloc[0]}")
if 'probabilities' in result:
    print(f"Wahrscheinlichkeiten:  {result['probabilities']}")

---
## Zusammenfassung der Verbesserungen gegenüber v1

| Aspekt | v1 (Random Forest) | v2 (optimiert) |
|---|---|---|
| **NaN-Handling** | Keine Imputation (6 NaN ignoriert) | Median-Imputation |
| **Feature-Selektion** | Nur Varianz-Filter (~15k Gene) | Varianz + ANOVA F-Score (2000 Gene) |
| **Scaling** | Keins | StandardScaler (nötig für LR/SVM) |
| **Modell** | Random Forest fix | 4 Kandidaten verglichen, bestes getunt |
| **Tuning** | RandomizedSearchCV (30 iter.) | RandomizedSearchCV (40 iter., balanced acc.) |
| **Evaluation** | Nur Accuracy | Accuracy + Balanced Accuracy + Macro F1 |
| **Visualisierung** | PCA (wenig Varianzanteil) | PCA + UMAP (bessere Cluster) |
| **Persistenz** | Nur Modell | Gesamte Pipeline (reproduzierbar) |
| **Inference** | Manuell | `predict_subtype()`-Funktion |
